### np.argsort

In [2]:

def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array detected: first row has shape {inner_shape}, "
                f"but row {i} has shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def flatten(a):
    if not isinstance(a, list):
        return [a]
    result = []
    for item in a:
        result.extend(flatten(item))
    return result

def mergesort_indices(lst, indices):
    if len(indices) <= 1:
        return indices[:]

    mid = len(indices) // 2
    left  = mergesort_indices(lst, indices[:mid])
    right = mergesort_indices(lst, indices[mid:])

    result = []
    i = 0
    j = 0
    while i < len(left) and j < len(right):
        if lst[left[i]] <= lst[right[j]]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])
    return result


def argsort_1d(lst, kind='mergesort'):
    if kind not in ('mergesort', 'stable', 'quicksort', 'heapsort'):
        raise ValueError(f"Unsupported sort kind: '{kind}'")
    indices = list(range(len(lst)))
    return mergesort_indices(lst, indices)

def argsort_last_axis(a, kind):
    if not isinstance(a, list):
        return 0

    if len(a) == 0:
        return []

    if not isinstance(a[0], list):
        return argsort_1d(a, kind)

    return [argsort_last_axis(sub, kind) for sub in a]


def np_argsort(a, axis=-1, kind='mergesort', stable=False):

    shape = get_shape(a)
    ndim  = len(shape)

    if ndim == 0:
        return 0

    if axis is None:
        flat = flatten(a)
        return argsort_1d(flat, kind)

    if axis < 0:
        axis += ndim
    if axis < 0 or axis >= ndim:
        raise ValueError(
            f"axis {axis - ndim if axis >= 0 else axis} is out of bounds "
            f"for array of dimension {ndim}"
        )

    if axis != ndim - 1:
        raise NotImplementedError("Only last axis (axis=-1) is supported for now")

    return argsort_last_axis(a, kind)

### test_cases 

In [3]:
print(np_argsort([30, 10, 20])) 
print(np_argsort(['b', 'a', 'c']))
print(np_argsort([5, 5, 5]))
print(np_argsort([[30, 10], [50, 20]]))
print(np_argsort([[[3, 1], [2, 0]], [[5, 4], [7, 6]]]))
print(np_argsort([[30, 10], [50, 20]], axis=None) )
print(np_argsort([[], []]))
print(np_argsort([99]) )
print(np_argsort(42) )

[1, 2, 0]
[1, 0, 2]
[0, 1, 2]
[[1, 0], [1, 0]]
[[[1, 0], [1, 0]], [[1, 0], [1, 0]]]
[1, 3, 0, 2]
[[], []]
[0]
0


In [4]:
np_argsort([1, 2, 3], kind='unknown') 

ValueError: Unsupported sort kind: 'unknown'

In [5]:
np_argsort([[1, 2], [3, 4]], axis=2)

ValueError: axis 0 is out of bounds for array of dimension 2

In [6]:
np_argsort([[1, 2], [3, 4]], axis=0)

NotImplementedError: Only last axis (axis=-1) is supported for now

In [7]:
np_argsort([[1, 2], [3]]) 

ValueError: Jagged array detected: first row has shape (2,), but row 1 has shape (1,)

In [8]:
np_argsort([1, 'a'])

TypeError: '<=' not supported between instances of 'int' and 'str'